In [3]:
import pandas as pd
df = pd.read_csv('../../data/walmart_transactions.csv', dtype=str)
df.store_id.value_counts()

store_id
FL-445     1819
FL-156     1808
FL-710     1804
FL-105     1800
FL-412     1787
FL-302     1778
FL-330     1773
FL-287     1749
FL-078     1746
FL-501     1728
FL-239     1727
FL-619     1695
FL_302      151
FL_239      140
FL_619      133
FL_287      131
FL_330      131
FL_445      130
FL_078      127
FL_156      123
FL_501      123
FL_412      123
FL_105      119
FL_710      118
FL287       102
fl-287      100
fl-501       96
FL239        96
FL445        95
fl-302       94
fl-156       93
fl-330       91
fl-105       89
fl-710       89
FL302        89
fl-239       88
fl-619       87
FL619        85
FL156        84
FL078        82
FL501        81
fl-445       80
 FL-330      78
 FL-302      78
fl-412       77
FL412        76
FL330        75
FL105        73
fl-078       73
 FL-710      72
 FL-619      68
FL710        66
 FL-156      62
 FL-287      59
 FL-078      57
 FL-412      56
 FL-105      53
 FL-445      53
 FL-239      52
 FL-501      50
Name: count, dtype: int64

In [ ]:
import pandas as pd

# View boolean series of duplicates
df.duplicated()

# Display the actual duplicate rows


0        False
1        False
2        False
3        False
4        False
         ...  
25557    False
25558    False
25559    False
25560    False
25561    False
Length: 25562, dtype: bool

In [7]:
# Near-duplicates: same store, time, SKU, quantity and price, but different transaction_id.
# Normalize the columns that differ only by formatting so the matches line up.
d = df.drop_duplicates().copy()
d['store_id'] = d.store_id.str.strip().str.upper().str.replace(r'^([A-Z]{2})[-_ ]?(\d+)$', r'\1-\2', regex=True)
d['unit_price'] = pd.to_numeric(d.unit_price.str.replace('$', '', regex=False))

key = ['store_id', 'timestamp', 'sku', 'quantity', 'unit_price']
near = d[d.duplicated(key, keep=False)].sort_values(key + ['transaction_id'])

# Show the original (un-normalized) rows so formatting differences stay visible.
# Each pair is its own two-row table, with a divider between pairs.
groups = list(near.groupby(key, sort=False).groups.values())
print(f'{len(near)} rows in {len(groups)} pairs\n')

for n, idx in enumerate(groups, start=1):
    pair = df.loc[idx]
    differs = [c for c in pair.columns if pair[c].nunique(dropna=False) > 1]
    print(f'PAIR {n} of {len(groups)}   columns that differ: {differs}')
    print(pair.to_string())
    print('-' * 120)

60 rows in 30 pairs

PAIR 1 of 30   columns that differ: ['transaction_id']
      transaction_id store_id            timestamp       sku    product_name            category quantity unit_price
23821      TXN010095   FL-078  2024-09-12 06:46:22  FLASH-01  LED Flashlight  Emergency Supplies        1      12.43
11665      TXN025507   FL-078  2024-09-12 06:46:22  FLASH-01  LED Flashlight  Emergency Supplies        1      12.43
------------------------------------------------------------------------------------------------------------------------
PAIR 2 of 30   columns that differ: ['transaction_id']
      transaction_id store_id          timestamp      sku product_name   category quantity unit_price
2267       TXN000687   FL-105  02-Sep-2024 16:55  TARP-HD      HD Tarp  emergency        2      20.48
10242      TXN025511   FL-105  02-Sep-2024 16:55  TARP-HD      HD Tarp  emergency        2      20.48
-------------------------------------------------------------------------------------------